# Day 040 — Exercise 4: narrate_eda_report

**What you'll build:** `narrate_eda_report(report, title, model) -> str` — take a full `eda_report` dict, extract the key facts into a compact summary, build a prompt, and return a 3-5 sentence executive summary.

**Why it matters:** A full EDA report dict has hundreds of numbers. You cannot dump the entire thing into a prompt — the LLM will lose focus. The skill is knowing what to include: shape, null count, column names, and the top category counts. That compact context gives the LLM enough to write a meaningful executive summary for a business reader.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import ollama
import pandas as pd
import io


def distribution_summary(df, col):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s):
        return {
            'count': int(s.count()), 'mean': round(float(s.mean()), 4),
            'std': round(float(s.std()), 4), 'min': float(s.min()),
            'q25': float(s.quantile(0.25)), 'median': float(s.quantile(0.50)),
            'q75': float(s.quantile(0.75)), 'max': float(s.max()),
            'null_count': int(s.isnull().sum()),
        }
    counts = s.value_counts()
    return {
        'count': int(s.count()), 'unique': int(s.nunique()),
        'top': str(counts.index[0]) if len(counts) else None,
        'top_freq': int(counts.iloc[0]) if len(counts) else 0,
        'null_count': int(s.isnull().sum()),
    }


def top_groups(df, group_col, value_col, n=5):
    return (
        df.groupby(group_col)[value_col]
        .agg(total='sum', mean='mean', count='count')
        .reset_index()
        .nlargest(n, 'total')
        .reset_index(drop=True)
    )


def correlation_summary(df, target_col):
    corr = df.select_dtypes(include='number').corr()[target_col].drop(target_col)
    result = pd.DataFrame({'feature': corr.index.tolist(), 'correlation': corr.values})
    result['_abs'] = result['correlation'].abs()
    result = result.sort_values('_abs', ascending=False).drop(columns='_abs')
    return result.reset_index(drop=True)


def eda_report(df):
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()
    return {
        'shape':           df.shape,
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': df[num_cols].describe().round(2).to_dict() if num_cols else {},
        'category_counts': {col: df[col].value_counts().to_dict() for col in cat_cols},
        'correlations':    df[num_cols].corr().round(4).to_dict() if len(num_cols) > 1 else {},
    }


RETAIL_CSV = (
    'order_id,product,category,region,price,quantity\n'
    '1,Widget,Electronics,North,25.0,10\n'
    '2,Gadget,Electronics,South,150.0,3\n'
    '3,Widget,Electronics,South,25.0,5\n'
    '4,Doohickey,Accessories,East,8.0,50\n'
    '5,Gadget,Electronics,East,150.0,7\n'
    '6,Widget,Electronics,East,25.0,4\n'
    '7,Doohickey,Accessories,North,8.0,20\n'
    '8,Gadget,Electronics,North,150.0,2\n'
    '9,Widget,Electronics,West,25.0,6\n'
    '10,Doohickey,Accessories,South,8.0,15\n'
    '11,Thingamajig,Accessories,North,200.0,1\n'
    '12,Thingamajig,Accessories,East,200.0,4'
)
SALES_DF = pd.read_csv(io.StringIO(RETAIL_CSV))
SALES_DF['revenue'] = SALES_DF['price'] * SALES_DF['quantity']


import json
import ollama

def summarize_column(col_name: str, stats: dict,
                     model: str = 'llama3.2') -> str:
    prompt = (
        f"You are a concise data analyst. Describe the column '{col_name}' "
        f"in 1-2 clear sentences for a non-technical reader.\n\n"
        f"Statistics:\n{json.dumps(stats, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()


import json
import ollama

def narrate_top_groups(groups_df, group_col: str, value_col: str,
                       model: str = 'llama3.2') -> str:
    records = groups_df.to_dict(orient='records')
    prompt = (
        f"You are a concise data analyst. Write 2-3 sentences about which "
        f"'{group_col}' groups have the highest '{value_col}' and what stands out.\n\n"
        f"Top groups by {value_col}:\n{json.dumps(records, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()


import json
import ollama

def narrate_correlations(corr_df, target_col: str,
                         model: str = 'llama3.2') -> str:
    records = corr_df.to_dict(orient='records')
    prompt = (
        f"You are a concise data analyst. Write 2-3 sentences explaining which "
        f"features correlate most with '{target_col}' and what this likely means.\n\n"
        f"Correlations with '{target_col}':\n{json.dumps(records, indent=2)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()

## Your Implementation

In [ ]:
def narrate_eda_report(report: dict, title: str = 'Dataset',
                       model: str = 'llama3.2') -> str:
    """
    Generate an executive summary from a full eda_report dict.

    Args:
        report — dict from eda_report() with shape/null_counts/etc.
        title  — human-readable dataset name for the prompt
        model  — Ollama model to use
    Returns:
        str    — 3-5 sentence executive summary
    """
    shape = report['shape']
    nulls = sum(report['null_counts'].values())
    # TODO: build a compact summary dict (rows, columns, nulls,
    #       list of numeric/categorical columns, numeric_stats,
    #       top 5 entries per category)
    # TODO: build prompt asking for a 3-5 sentence executive summary
    # TODO: call ollama.chat and return stripped content
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0
    _result = None

    # Check 1: function defined
    try:
        assert 'narrate_eda_report' in globals()
        passed += 1; print('\u2705 Check 1: narrate_eda_report is defined')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: returns a string (Ollama call)
    try:
        _report = eda_report(SALES_DF)
        _result = narrate_eda_report(_report, title='Retail Sales')
        assert isinstance(_result, str), \
            f'expected str, got {type(_result).__name__}'
        passed += 1; print('\u2705 Check 2: returns a string')
    except Exception as e:
        print(f'\u274c Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 3: response is non-empty
    try:
        assert len(_result.strip()) > 0, 'response is empty'
        passed += 1; print('\u2705 Check 3: response is non-empty')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: response is longer than a single-column summary (> 100 chars)
    try:
        _n = len(_result.strip())
        assert _n > 100, \
            f'response too short ({_n} chars) — executive summary should be > 100 chars'
        passed += 1; print(f'\u2705 Check 4: response is {_n} chars (full EDA summary)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: title parameter is accepted (no TypeError)
    try:
        _result2 = narrate_eda_report(_report, title='Q4 Sales Report',
                                      model='llama3.2')
        assert isinstance(_result2, str) and len(_result2.strip()) > 0
        passed += 1; print('\u2705 Check 5: title and model parameters accepted')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import json
import ollama

def narrate_eda_report(report: dict, title: str = 'Dataset',
                       model: str = 'llama3.2') -> str:
    shape = report['shape']
    nulls = sum(report['null_counts'].values())
    summary = {
        'dataset':             title,
        'rows':                shape[0],
        'columns':             shape[1],
        'total_nulls':         nulls,
        'numeric_columns':     list(report['numeric_summary'].keys()),
        'categorical_columns': list(report['category_counts'].keys()),
        'numeric_stats':       report['numeric_summary'],
        'top_categories':      {
            col: dict(list(counts.items())[:5])
            for col, counts in report['category_counts'].items()
        },
    }
    prompt = (
        "You are a data analyst. Write a 3-5 sentence executive summary of this "
        "dataset for a business audience. Highlight key patterns, data quality, "
        "and notable findings.\n\n"
        f"EDA Report:\n{json.dumps(summary, indent=2, default=str)}"
    )
    resp = ollama.chat(model=model,
                       messages=[{"role": "user", "content": prompt}])
    return resp["message"]["content"].strip()
```

</details>